# LOB-Quant-HFT: Exploratory Data Analysis

This notebook covers:
1. Synthetic LOB data generation & inspection
2. Mid-price and spread dynamics
3. Label distribution (class imbalance analysis)
4. Microstructure feature distributions (OFI, VOI, depth imbalance)
5. Feature correlation heatmap
6. Autocorrelation of returns and OFI

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.data.preprocess import generate_synthetic_lob, label_from_mid_price
from src.utils.config import set_seed

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
set_seed(42)
print('Imports OK')

## 1. Generate Synthetic LOB Data

In [ ]:
N_TICKS   = 20_000
N_LEVELS  = 10
TICK_SIZE = 0.01

X, y = generate_synthetic_lob(
    n_ticks=N_TICKS,
    n_levels=N_LEVELS,
    tick_size=TICK_SIZE,
    seed=42,
)

# Build column names: ask_p1, ask_v1, bid_p1, bid_v1, ...
cols = []
for i in range(1, N_LEVELS + 1):
    cols += [f'ask_p{i}', f'ask_v{i}', f'bid_p{i}', f'bid_v{i}']

df = pd.DataFrame(X, columns=cols)
df['label'] = y
df['mid']   = (df['ask_p1'] + df['bid_p1']) / 2
df['spread']= df['ask_p1'] - df['bid_p1']

print(df.head())
print(f'\nShape: {df.shape}')
print(f'Label distribution:\n{df["label"].value_counts().sort_index()}')

## 2. Mid-Price & Spread Dynamics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Mid-price
axes[0].plot(df['mid'].values[:2000], lw=0.8, color='#2196F3')
axes[0].set_ylabel('Mid Price')
axes[0].set_title('LOB Mid-Price (first 2000 ticks)')

# Spread
axes[1].fill_between(range(2000), df['spread'].values[:2000],
                     alpha=0.5, color='#FF5722')
axes[1].set_ylabel('Spread')
axes[1].set_title('Bid-Ask Spread')

# Labels
color_map = {0: '#F44336', 1: '#9E9E9E', 2: '#4CAF50'}
label_names = {0: 'Down', 1: 'Stat', 2: 'Up'}
for lbl, color in color_map.items():
    mask = df['label'].values[:2000] == lbl
    axes[2].scatter(np.where(mask)[0], [lbl]*mask.sum(),
                    c=color, s=2, label=label_names[lbl])
axes[2].set_ylabel('Label')
axes[2].set_yticks([0, 1, 2])
axes[2].set_yticklabels(['Down', 'Stat', 'Up'])
axes[2].legend(loc='upper right', markerscale=5)
axes[2].set_title('Label Series')
axes[2].set_xlabel('Tick')

plt.tight_layout()
plt.savefig('../results/mid_price_dynamics.png', bbox_inches='tight')
plt.show()

## 3. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts().sort_index()
bars = axes[0].bar(['Down', 'Stationary', 'Up'], counts.values,
                    color=['#F44336', '#9E9E9E', '#4CAF50'], edgecolor='black', lw=0.5)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{cnt:,}\n({cnt/N_TICKS:.1%})', ha='center', fontsize=10)
axes[0].set_title('Label Counts')
axes[0].set_ylabel('Count')

# Rolling label frequency
window = 500
for lbl, color, name in [(0, '#F44336', 'Down'), (1, '#9E9E9E', 'Stat'), (2, '#4CAF50', 'Up')]:
    freq = (df['label'] == lbl).rolling(window).mean()
    axes[1].plot(freq, color=color, lw=1.2, label=name)
axes[1].set_title(f'Rolling Label Frequency (window={window})')
axes[1].set_ylabel('Frequency')
axes[1].set_xlabel('Tick')
axes[1].legend()
axes[1].axhline(1/3, color='black', ls='--', lw=0.8, label='1/3')

plt.tight_layout()
plt.savefig('../results/label_distribution.png', bbox_inches='tight')
plt.show()

## 4. Microstructure Features

In [ ]:
from src.features.microstructure import (
    order_flow_imbalance,
    volume_order_imbalance,
    depth_imbalance,
    queue_imbalance,
    relative_spread,
    realized_volatility,
)

ofi  = order_flow_imbalance(df, level=1, window=5)
voi  = volume_order_imbalance(df, n_levels=N_LEVELS)
di   = depth_imbalance(df, n_levels=N_LEVELS, weights='linear')
qi   = queue_imbalance(df, level=1)
rs   = relative_spread(df)
rv   = realized_volatility(df, window=20)

features = pd.DataFrame({
    'OFI':             ofi,
    'VOI':             voi,
    'DepthImbalance':  di,
    'QueueImbalance':  qi,
    'RelSpread':       rs,
    'RealizedVol':     rv,
})

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, features.columns):
    vals = features[col].values[:3000]
    ax.plot(vals, lw=0.6, alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('Tick')

plt.suptitle('Microstructure Features (first 3000 ticks)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/microstructure_features.png', bbox_inches='tight')
plt.show()
print(features.describe().round(4))

## 5. Feature Correlation Heatmap

In [ ]:
# Sample raw LOB features (first 4 per level + microstructure)
raw_cols = ['ask_p1', 'ask_v1', 'bid_p1', 'bid_v1',
            'ask_p2', 'ask_v2', 'bid_p2', 'bid_v2']
corr_df = pd.concat([
    df[raw_cols].reset_index(drop=True),
    features.reset_index(drop=True),
], axis=1)

corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../results/feature_correlation.png', bbox_inches='tight')
plt.show()

## 6. LOB Depth Snapshot

In [ ]:
# Visualise a single LOB snapshot
t = 1000
bid_prices  = [df[f'bid_p{i}'].iloc[t] for i in range(1, N_LEVELS + 1)]
bid_volumes = [df[f'bid_v{i}'].iloc[t] for i in range(1, N_LEVELS + 1)]
ask_prices  = [df[f'ask_p{i}'].iloc[t] for i in range(1, N_LEVELS + 1)]
ask_volumes = [df[f'ask_v{i}'].iloc[t] for i in range(1, N_LEVELS + 1)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(bid_prices, bid_volumes, height=TICK_SIZE * 0.8,
        color='#4CAF50', alpha=0.8, label='Bid')
ax.barh(ask_prices, [-v for v in ask_volumes], height=TICK_SIZE * 0.8,
        color='#F44336', alpha=0.8, label='Ask')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Volume')
ax.set_ylabel('Price')
ax.set_title(f'LOB Depth Snapshot at tick {t}')
ax.legend()
plt.tight_layout()
plt.savefig('../results/lob_snapshot.png', bbox_inches='tight')
plt.show()

## 7. Return Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

log_ret = np.diff(np.log(df['mid'].values + 1e-10))
abs_ret = np.abs(log_ret)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf( log_ret, lags=50, ax=axes[0,0], title='ACF: Log Returns')
plot_pacf(log_ret, lags=50, ax=axes[0,1], title='PACF: Log Returns')
plot_acf( abs_ret, lags=50, ax=axes[1,0], title='ACF: |Log Returns| (Volatility Clustering)')
plot_pacf(abs_ret, lags=50, ax=axes[1,1], title='PACF: |Log Returns|')

plt.tight_layout()
plt.savefig('../results/return_autocorrelation.png', bbox_inches='tight')
plt.show()

## 8. OFI vs Future Return

In [ ]:
future_ret = np.roll(log_ret, -10)
future_ret[-10:] = 0
ofi_clipped = np.clip(ofi[:len(log_ret)], -500, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(ofi_clipped[::20], future_ret[::20],
                alpha=0.15, s=5, color='#3F51B5')
axes[0].set_xlabel('OFI (clipped)')
axes[0].set_ylabel('Future 10-tick log return')
axes[0].set_title('OFI vs Future Return')

# OFI distribution by label
for lbl, color, name in [(0,'#F44336','Down'), (1,'#9E9E9E','Stat'), (2,'#4CAF50','Up')]:
    mask = df['label'].values[:len(ofi_clipped)] == lbl
    axes[1].hist(ofi_clipped[mask], bins=60, alpha=0.5,
                 color=color, label=name, density=True)
axes[1].set_xlabel('OFI')
axes[1].set_ylabel('Density')
axes[1].set_title('OFI Distribution by Label')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/ofi_analysis.png', bbox_inches='tight')
plt.show()

## Summary

| Feature | Insight |
|---|---|
| Mid-price | Random walk with ~symmetric returns |
| Label distribution | Mild class imbalance — use focal loss |
| OFI | Slight positive correlation with future return |
| Spread | Near-constant in synthetic data (use real data for realistic dynamics) |
| Volatility clustering | Present in absolute returns (ARCH effects) |
| Feature correlations | Price levels highly correlated; volume features less so |